In [2]:
import pandas as pd
import numpy as np

# Football Match Analysis and Result Prediction

## Introduction

Football match prediction is a challenging machine learning problem because match outcomes depend on many factors including team strength, recent form, home advantage, injuries, and tactical decisions.

The goal of this project is to analyze historical football match data and build machine learning models capable of predicting match outcomes.

The notebook includes:

- Data collection and preprocessing
- Exploratory Data Analysis (EDA)
- Feature engineering
- Machine learning models
- Model evaluation
- Match prediction examples

# Mathematical Background

Football match prediction can be formulated as a multiclass classification problem.

Input features:

x = [x₁, x₂, ..., xₙ]

where x represents information about both teams.

The prediction function is:

f(x) → y

where:

- 0 = Away Win
- 1 = Draw
- 2 = Home Win

Evaluation metrics:

### Accuracy

Accuracy = Correct Predictions / Total Predictions

### Precision

Precision = TP / (TP + FP)

### Recall

Recall = TP / (TP + FN)

### F1 Score

F1 = 2 × (Precision × Recall) / (Precision + Recall)

These metrics allow us to compare different machine learning models.

# Dataset Description

The dataset is obtained from Football-Data.co.uk.

Football-Data provides historical football match results, team statistics, and betting odds in CSV format.

For this project, matches from a selected league and season range are used.

Examples of available leagues:

- Premier League
- La Liga
- Bundesliga
- Serie A

The target variable is the Full Time Result (FTR):

- H = Home Win
- D = Draw
- A = Away Win

In [ ]:
df = pd.read_csv(
    "https://www.football-data.co.uk/mmz4281/2425/E0.csv"
)

print(df.head())
print(df.columns)

In [ ]:
df["Date"] = pd.to_datetime(df["Date"], dayfirst=True)

# Sort chronologically

df = df.sort_values("Date")

# Keep relevant columns

columns = [
    "Date",
    "HomeTeam",
    "AwayTeam",
    "FTHG",
    "FTAG",
    "FTR"
]

df = df[columns]

# Target encoding

mapping = {
    "A": 0,
    "D": 1,
    "H": 2
}

df["Target"] = df["FTR"].map(mapping)

df.head()

# Feature Engineering

Raw football data is usually not sufficient for accurate predictions.

Instead of using information from the current match, we create features based only on previous matches.

Rolling statistics provide a good estimate of current team strength.

The following features are created:

- Average goals scored
- Average goals conceded
- Recent win rate
- Form difference between teams

These features simulate how analysts evaluate team performance before a match.

In [ ]:
df["HomeGoalsAvg"] = (
    df.groupby("HomeTeam")["FTHG"]
      .transform(lambda x: x.shift().rolling(5, min_periods=1).mean())
)

# Away team average goals scored

df["AwayGoalsAvg"] = (
    df.groupby("AwayTeam")["FTAG"]
      .transform(lambda x: x.shift().rolling(5, min_periods=1).mean())
)

# Goal difference feature

df["FormDifference"] = (
    df["HomeGoalsAvg"] -
    df["AwayGoalsAvg"]
)

df = df.dropna()

df.head()

In [ ]:
plt.figure(figsize=(8,5))
sns.countplot(x="FTR", data=df)
plt.title("Match Outcome Distribution")
plt.show()

plt.figure(figsize=(8,5))
sns.histplot(df["FTHG"], bins=10)
plt.title("Home Goals Distribution")
plt.show()

plt.figure(figsize=(8,5))

corr = df[
    [
        "FTHG",
        "FTAG",
        "HomeGoalsAvg",
        "AwayGoalsAvg",
        "FormDifference"
    ]
].corr()

sns.heatmap(
    corr,
    annot=True,
    cmap="coolwarm"
)

plt.title("Correlation Heatmap")
plt.show()